### Carregando Dados

In [1]:
import pandas as pd

In [7]:
X_train = pd.read_parquet('../Datasets/Split/X_train_raw.parquet')
X_test = pd.read_parquet('../Datasets/Split/X_test_raw.parquet')
y_train = pd.read_parquet('../Datasets/Split/y_train.parquet')
y_test = pd.read_parquet('../Datasets/Split/y_test.parquet')

### One Hot Encoding

In [5]:
df = pd.get_dummies(
    df,
    columns=['SG_UF'],
    drop_first=False,
    dtype=int
)

### AutoML com AutoGluon

In [11]:
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from tqdm import tqdm
import os
import shutil


/home/guilherme_fernandes/anaconda3/envs/aprendizagem_automatica/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
def prepare_autogluon_data(X, y, target_column):
    data = X.copy()
    data[target_column] = y.values
    return data

In [13]:
def run_automl(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    target_column,
    id_column=None,
    output_path="autogluon_regression",
    time_limit=600,
    overwrite=True
):
    if overwrite and os.path.exists(output_path):
        shutil.rmtree(output_path)

    if id_column is not None:
        X_train = X_train.drop(columns=[id_column], errors="ignore")
        X_val = X_val.drop(columns=[id_column], errors="ignore")
        X_test = X_test.drop(columns=[id_column], errors="ignore")

    train_data = prepare_autogluon_data(X_train, y_train, target_column)
    val_data = prepare_autogluon_data(X_val, y_val, target_column)

    train_full = pd.concat(
        [train_data, val_data],
        ignore_index=True
    )

    test_data = prepare_autogluon_data(X_test, y_test, target_column)

    predictor = TabularPredictor(
        label=target_column,
        problem_type="regression",
        eval_metric="root_mean_squared_error",
        path=output_path
    )

    predictor.fit(
        train_data=train_full,
        time_limit=time_limit,
        presets="best_quality"
    )

    X_test_final = test_data.drop(columns=[target_column])

    predictions = predictor.predict(X_test_final)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    metrics = {
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }

    leaderboard = predictor.leaderboard(
        test_data,
        silent=True
    )

    return predictor, metrics, leaderboard, predictions

In [14]:
predictor, metrics, leaderboard, predictions = run_automl(
    X_train=X_train_scaled,
    y_train=y_train,
    X_val=X_val_scaled,
    y_val=y_val,
    X_test=X_test_scaled,
    y_test=y_test,
    target_column="TAXA_CRE_INT",
    id_column="NO_MUNICIPIO",
    output_path="autogluon_regression",
    time_limit=3600
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 11 01:32:26 UTC 2026
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       8.70 GB / 15.23 GB (57.1%)
Disk Space Avail:   48.56 GB / 182.28 GB (26.6%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `nu

(_ray_fit pid=8775) [1000]	valid_set's rmse: 0.306164


(_dystack pid=1337) 	-0.3089	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	1.65s	 = Training   runtime
(_dystack pid=1337) 	0.04s	 = Validation runtime
(_dystack pid=1337) Fitting model: NeuralNetTorch_r22_BAG_L1 ... Training model for up to 534.39s of the 833.26s of remaining time.
(_dystack pid=1337) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.02%)
(_ray_fit pid=9239) /home/guilherme_fernandes/anaconda3/envs/aprendizagem_automatica/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
(_ray_fit pid=9239)   warnings.warn(
(_dystack pid=1337) 	-0.3315	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	8.28s	 = Training   runtime
(_dystack pid=1337) 	0.16s	 = Validation runtime

(_ray_fit pid=20548) [1000]	valid_set's rmse: 0.281798 [repeated 5x across cluster]


(_dystack pid=1337) 	-0.309	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	5.42s	 = Training   runtime
(_dystack pid=1337) 	0.07s	 = Validation runtime
(_dystack pid=1337) Fitting model: RandomForest_r39_BAG_L1 ... Training model for up to 387.99s of the 686.86s of remaining time.
(_dystack pid=1337) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=16, gpus=0
(_ray_fit pid=20047) No improvement since epoch 3: early stopping [repeated 7x across cluster]
(_dystack pid=1337) 	-0.3187	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	0.71s	 = Training   runtime
(_dystack pid=1337) 	0.06s	 = Validation runtime
(_dystack pid=1337) Fitting model: CatBoost_r167_BAG_L1 ... Training model for up to 387.20s of the 686.06s of remaining time.
(_dystack pid=1337) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=5.74%)
(_dystack pid=1337) 	-0.3108	 = Validation 

...

(_dystack pid=1337) 	-0.3021	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	4.31s	 = Training   runtime
(_dystack pid=1337) 	0.08s	 = Validation runtime
(_dystack pid=1337) Fitting model: NeuralNetTorch_r22_BAG_L2 ... Training model for up to 222.82s of the 222.73s of remaining time.
(_dystack pid=1337) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.06%)
(_ray_fit pid=57504) /home/guilherme_fernandes/anaconda3/envs/aprendizagem_automatica/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
(_ray_fit pid=57504)   warnings.warn(
(_dystack pid=1337) 	-0.3334	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	6.71s	 = Training   runtime
(_dystack pid=1337) 	0.08s	 = Validation runti

(_ray_fit pid=60447) [1000]	valid_set's rmse: 0.286658 [repeated 10x across cluster]
(_ray_fit pid=60445) [2000]	valid_set's rmse: 0.284905 [repeated 9x across cluster]
(_ray_fit pid=60442) [4000]	valid_set's rmse: 0.309318 [repeated 7x across cluster]
(_ray_fit pid=60441) [6000]	valid_set's rmse: 0.290396 [repeated 6x across cluster]
(_ray_fit pid=60445) [6000]	valid_set's rmse: 0.2849 [repeated 6x across cluster]
(_ray_fit pid=60445) [8000]	valid_set's rmse: 0.2849 [repeated 5x across cluster]


(_dystack pid=1337) 	-0.3004	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	37.66s	 = Training   runtime
(_dystack pid=1337) 	0.89s	 = Validation runtime
(_dystack pid=1337) Fitting model: NeuralNetFastAI_r145_BAG_L2 ... Training model for up to 120.33s of the 120.24s of remaining time.
(_dystack pid=1337) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.10%)
(_ray_fit pid=60949) No improvement since epoch 3: early stopping
(_dystack pid=1337) 	-0.3056	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	6.01s	 = Training   runtime
(_dystack pid=1337) 	0.08s	 = Validation runtime
(_dystack pid=1337) Fitting model: XGBoost_r89_BAG_L2 ... Training model for up to 113.11s of the 113.03s of remaining time.
(_dystack pid=1337) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.80%)
(_dystack pid=13

(_ray_fit pid=66741) [1000]	valid_set's rmse: 0.287952 [repeated 2x across cluster]


(_dystack pid=1337) 	-0.3071	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1337) 	6.03s	 = Training   runtime
(_dystack pid=1337) 	0.05s	 = Validation runtime
(_dystack pid=1337) Fitting model: NeuralNetFastAI_r143_BAG_L2 ... Training model for up to 14.76s of the 14.68s of remaining time.
(_dystack pid=1337) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.09%)
(_ray_fit pid=66256) /home/guilherme_fernandes/anaconda3/envs/aprendizagem_automatica/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning. [repeated 7x across cluster]
(_ray_fit pid=66256)   warnings.warn( [repeated 7x across cluster]
(_ray_fit pid=67215) No improvement since epoch 5: early stopping
(_dystack pid=1337) 	-0.3064	 = Validation score 

In [15]:
leaderboard

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost_r69_BAG_L1,-0.305733,-0.310358,root_mean_squared_error,0.010092,0.006946,1.518379,0.010092,0.006946,1.518379,1,True,33
1,NeuralNetFastAI_r160_BAG_L1,-0.306288,-0.307994,root_mean_squared_error,0.116079,0.094204,5.070493,0.116079,0.094204,5.070493,1,True,75
2,LightGBM_r96_BAG_L1,-0.306805,-0.309427,root_mean_squared_error,0.024602,0.036252,1.041376,0.024602,0.036252,1.041376,1,True,15
3,XGBoost_r89_BAG_L1,-0.306846,-0.313833,root_mean_squared_error,0.032134,0.024283,0.903103,0.032134,0.024283,0.903103,1,True,25
4,CatBoost_r60_BAG_L1,-0.306888,-0.310666,root_mean_squared_error,0.016732,0.006956,2.009156,0.016732,0.006956,2.009156,1,True,76
...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,NeuralNetTorch_r31_BAG_L1,-0.337604,-0.332468,root_mean_squared_error,0.062397,0.065753,5.637991,0.062397,0.065753,5.637991,1,True,61
103,NeuralNetTorch_r76_BAG_L1,-0.339503,-0.329801,root_mean_squared_error,0.051714,0.067739,4.625883,0.051714,0.067739,4.625883,1,True,86
104,NeuralNetTorch_r19_BAG_L1,-0.340220,-0.331017,root_mean_squared_error,0.045204,0.066069,4.087522,0.045204,0.066069,4.087522,1,True,101
105,NeuralNetTorch_r89_BAG_L1,-0.341080,-0.336418,root_mean_squared_error,0.066197,0.105615,8.532468,0.066197,0.105615,8.532468,1,True,106


In [16]:
metrics

{'rmse': np.float64(0.30747117806625385),
 'mae': 0.23923081665285262,
 'r2': 0.26678992693168113}

In [17]:
leaderboard.head()

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost_r69_BAG_L1,-0.305733,-0.310358,root_mean_squared_error,0.010092,0.006946,1.518379,0.010092,0.006946,1.518379,1,True,33
1,NeuralNetFastAI_r160_BAG_L1,-0.306288,-0.307994,root_mean_squared_error,0.116079,0.094204,5.070493,0.116079,0.094204,5.070493,1,True,75
2,LightGBM_r96_BAG_L1,-0.306805,-0.309427,root_mean_squared_error,0.024602,0.036252,1.041376,0.024602,0.036252,1.041376,1,True,15
3,XGBoost_r89_BAG_L1,-0.306846,-0.313833,root_mean_squared_error,0.032134,0.024283,0.903103,0.032134,0.024283,0.903103,1,True,25
4,CatBoost_r60_BAG_L1,-0.306888,-0.310666,root_mean_squared_error,0.016732,0.006956,2.009156,0.016732,0.006956,2.009156,1,True,76


In [18]:
leaderboard.to_csv('leaderboard_automl.csv', index=False)